In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# 1. Încărcare datsetul neprocesat
df = pd.read_csv('dataset_min_main.csv')

# Tratarea valorilor lipsă
df['Job Title'] = df['Job Title'].fillna('')
df['Job Description'] = df['Job Description'].fillna('')
df['skills'] = df['skills'].fillna('')
df['Responsibilities'] = df['Responsibilities'].fillna('')
df['Role'] = df['Role'].fillna('Unknown')

# Creăm coloana combinată pentru antrenare direct în df
df['Train_Text'] = df['Job Title'] + " " + df['Job Description'] + " " + df['skills'] + " " + df['Responsibilities']

# Creăm coloana pentru testare direct în df
df['Test_CV_Simulated'] = df['skills']

# 2. Split-ul se face pe întregul DataFrame pentru a păstra toate coloanele
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Parametrii optimizati
max_features_options = [500, 1000]
ngram_options = [(1, 1), (1, 2)]
neighbors_options = [3, 5, 7]

best_precision = 0.0
best_config = {}


for max_f in max_features_options:
    for ngram in ngram_options:

        # Fit strict pe textul complet de antrenare
        tfidf = TfidfVectorizer(stop_words='english', max_features=max_f, ngram_range=ngram)
        X_train_tfidf = tfidf.fit_transform(train_df['Train_Text']).toarray()

        # Transformăm textul de test
        X_test_tfidf = tfidf.transform(test_df['Test_CV_Simulated']).toarray()

        for n_neigh in neighbors_options:
            knn_text = NearestNeighbors(n_neighbors=n_neigh, metric='cosine', algorithm='brute')
            knn_text.fit(X_train_tfidf)

            distances, indices = knn_text.kneighbors(X_test_tfidf)

            hits = 0
            k_eval = 5  # Precision

            for i in range(len(X_test_tfidf)):

                actual_role = test_df.iloc[i]['Role']

                predicted_neighbor_indices = indices[i][:k_eval]
                predicted_roles = train_df.iloc[predicted_neighbor_indices]['Role'].values

                if actual_role in predicted_roles:
                    hits += 1

            precision = hits / len(X_test_tfidf)
            print(f"Max Features: {max_f:4} | N-gram: {str(ngram):6} | Vecini KNN: {n_neigh} -> Precision@5: {precision*100:.2f}%")

            if precision > best_precision:
                best_precision = precision
                best_config = {'max_features': max_f, 'ngram_range': ngram, 'n_neighbors': n_neigh}


print(f" Configurația Optimă Text NLP: {best_config}")
print(f"Precizia Științifică: {best_precision * 100:.2f}%")

# 4. Antrenarea finală pe toate datele
final_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=best_config['max_features'],
    ngram_range=best_config['ngram_range']
)
X_final_tfidf = final_tfidf.fit_transform(df['Train_Text']).toarray()

final_knn_text = NearestNeighbors(n_neighbors=best_config['n_neighbors'], metric='cosine', algorithm='brute')
final_knn_text.fit(X_final_tfidf)

# 5. Salvarea finala
joblib.dump(final_tfidf, "tfidf_vectorizer.pkl")
joblib.dump(final_knn_text, "recommender_text.pkl")
print(" Fisierele 'tfidf_vectorizer.pkl' și 'recommender_text.pkl' au fost generate cu succes!")

Max Features:  500 | N-gram: (1, 1) | Vecini KNN: 3 -> Precision@5: 89.59%
Max Features:  500 | N-gram: (1, 1) | Vecini KNN: 5 -> Precision@5: 89.59%
Max Features:  500 | N-gram: (1, 1) | Vecini KNN: 7 -> Precision@5: 89.59%
Max Features:  500 | N-gram: (1, 2) | Vecini KNN: 3 -> Precision@5: 87.42%
Max Features:  500 | N-gram: (1, 2) | Vecini KNN: 5 -> Precision@5: 87.42%
Max Features:  500 | N-gram: (1, 2) | Vecini KNN: 7 -> Precision@5: 87.42%
Max Features: 1000 | N-gram: (1, 1) | Vecini KNN: 3 -> Precision@5: 91.65%
Max Features: 1000 | N-gram: (1, 1) | Vecini KNN: 5 -> Precision@5: 91.65%
Max Features: 1000 | N-gram: (1, 1) | Vecini KNN: 7 -> Precision@5: 91.65%
Max Features: 1000 | N-gram: (1, 2) | Vecini KNN: 3 -> Precision@5: 88.39%
Max Features: 1000 | N-gram: (1, 2) | Vecini KNN: 5 -> Precision@5: 88.39%
Max Features: 1000 | N-gram: (1, 2) | Vecini KNN: 7 -> Precision@5: 88.39%
 Configurația Optimă Text NLP: {'max_features': 1000, 'ngram_range': (1, 1), 'n_neighbors': 3}
Preci